# 7.4 n-step SARSA 算法

> n-step SARSA 是 **SARSA（TD(1)）与蒙特卡洛（MC）方法之间的统一桥梁**，通过引入超参数 $n$ 控制回望步数，在**偏差（Bias）与方差（Variance）之间取得灵活平衡**。
>
> **普通 SARSA（n=1）** 的更新目标仅向前看 1 步：
>
> $$G_t^{(1)} = r_{t+1} + \gamma Q(s_{t+1}, a_{t+1})$$
>
> **n-step SARSA** 将回望窗口扩展至 $n$ 步，TD 目标（n-step 回报）变为：
>
> $$G_t^{(n)} = r_{t+1} + \gamma r_{t+2} + \cdots + \gamma^{n-1} r_{t+n} + \gamma^n Q(s_{t+n}, a_{t+n})$$
>
> Q 值更新公式：
>
> $$Q(s_t, a_t) \leftarrow Q(s_t, a_t) - \alpha \left[Q(s_t, a_t) - G_t^{(n)}\right]$$
>
> 四种典型情形的对比如下：
>
> ---

| $n$ 取值 | 等价方法 | TD 目标 | 偏差 | 方差 |
|---|---|---|---|---|
| $n=1$，采样 $a'$ | 普通 SARSA（TD(1)） | $r_{t+1} + \gamma Q(s_{t+1}, a_{t+1})$ | 高（Bootstrap 引入） | 低 |
| $n=1$，期望 $a'$ | Expected SARSA | $r_{t+1} + \gamma \sum_{a'}\pi(a'\|s_{t+1})Q(s_{t+1},a')$ | 高（Bootstrap 引入） | **更低**（消除 $a'$ 采样噪声） |
| $1 < n < \infty$ | n-step SARSA | $\sum_{k=0}^{n-1}\gamma^k r_{t+k+1} + \gamma^n Q(s_{t+n}, a_{t+n})$ | 中 | 中 |
| $n \to \infty$ | 蒙特卡洛（MC） | $\sum_{k=0}^{T-t-1}\gamma^k r_{t+k+1}$（完整回报） | 低（无 Bootstrap） | 高 |

> ---
>
> **适当选择 $n$ 是 n-step 方法的核心**：$n$ 越大，Bootstrap 引入的偏差越小，但需等待更多步才能更新，方差也随之增大；$n=1$ 方差最低但偏差最大。实践中 $n \in [3, 10]$ 往往取得最佳权衡。

## 一、导入库与环境初始化

In [2]:
# 导入 NumPy 库，用于数组操作（Q 表、策略矩阵等）
import numpy as np
# 导入 random 模块，用于随机选择初始状态和动作
import random
# 导入 importlib 标准库，用于按文件路径动态加载模块
import importlib.util
# 导入 os 模块，用于拼接当前目录与文件名
import os
# 导入 time 模块（备用，可控制训练展示节奏）
import time
# 导入 Jupyter 显示控制函数，训练时清除输出以实时刷新展示
from IPython.display import clear_output

# 环境文件名 “02.1.ModelFree_Env_GridWorldV2.py” 以数字开头，
# Python 无法直接 import，需通过 importlib 按路径加载
_env_path = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "02.1.ModelFree_Env_GridWorldV2.py")  # 环境文件的绝对路径（str）
_spec = importlib.util.spec_from_file_location(
    "GridWorld_v2", _env_path)   # 构造模块规格对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)  # 根据规格创建模块对象
_spec.loader.exec_module(GridWorld_v2)                  # 执行模块代码，完成加载

In [3]:
# 设置网格世界行数（必须与 desc 字符串行数一致）
rows = 5
# 设置网格世界列数（必须与 desc 每行字符数一致）
columns = 5

# 创建 5×5 GridWorld 环境实例
# forbiddenAreaScore=-10: 进入障碍格（'#'）的即时惩罚
# score=1: 到达目标格（'T'）的即时正奖励
# desc: 地图描述，'.'=普通格，'#'=障碍格，'T'=目标格
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10, score=1,
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."])

# 显示网格地图
gridworld.show()

# 初始化状态价值向量（全零，用于展示）
# shape: (25,)——每个元素对应一个状态（0~24）的价值估计
value = np.zeros(rows * columns)

# 初始化 Q 表（动作价值矩阵，全零）
# shape: (25, 5)——行=状态索引[0~24]，列=5种动作的 Q 值
qtable = np.zeros((rows * columns, 5))

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️


## 二、n-step SARSA 算法实现

In [4]:
def n_step_SARSA(gridworld: GridWorld_v2.GridWorld_v2,
                sarsaStep=5,
                epsilon=0.5,
                trajectorySteps=-1,
                learning_rate=0.001,
                final_epsilon=0.01,
                num_episodes=600,
                gamma=0.9) -> GridWorld_v2.GridWorld_v2:
    '''
    n-step SARSA 算法。

    在 SARSA 基础上，将 1 步 TD 误差扩展为 n 步累积回报，
    兼顾 TD 的低方差和 MC 的无偏性，可通过调整 n 来平衡偏差与方差。

    参数:
        gridworld (GridWorld_v2): GridWorld 环境实例
        sarsaStep (int): n-step 中的步数 n（n=1 退化为 SARSA，n→∞ 退化为 MC）
        epsilon (float): epsilon-greedy 初始探索率（0~1），越大越随机
        trajectorySteps (int): 每条轨迹最大步数；-1 表示到目标自动停止
        learning_rate (float): TD 学习率，控制 Q 值更新幅度
        final_epsilon (float): epsilon 的最小值，防止完全停止探索
        num_episodes (int): 训练轮数（每轮生成一条轨迹）
        gamma (float): 折扣因子，取值 (0,1]

    返回值:
        GridWorld_v2.GridWorld_v2: 训练后的环境对象（可查看最终策略）
    '''
    # 初始化状态价值向量（全零，仅用于展示）
    # shape: (25,)——每个元素对应一个状态的价值
    state_value = np.zeros((rows * columns))

    # 初始化动作价值矩阵 Q（全零）
    # shape: (25, 5)——行=状态[0~24]，列=5种动作的 Q 值
    action_value = np.zeros((rows * columns, 5))

    # 初始化随机确定性策略（独热编码）
    # shape: (25, 5)——每行只有一个位置为 1
    policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]

    for episode in range(num_episodes):  # 训练主循环
        print("episode", f"{episode}/{num_episodes}")  # 打印当前轮次

        # 线性衰减 epsilon（逐步从探索转向利用）
        if epsilon > final_epsilon:
            epsilon -= 0.001   # 每轮减少 0.001
        else:
            epsilon = final_epsilon  # 不低于最小探索率

        # 计算 epsilon-greedy 概率分配
        # p1：贪心动作的选取概率
        p1 = 1 - epsilon * (4/5)
        # p0：非贪心动作各自的选取概率
        p0 = epsilon / 5
        # 概率映射字典
        d = {1: p1, 0: p0}
        print("p1", p1, "p0", p0)  # 打印当前概率值

        # 将确定性策略转为 epsilon-greedy 概率分布
        # shape: (25, 5)——每行是一个状态的动作概率分布
        policy_epsilon = np.vectorize(d.get)(policy)

        # 初始化状态访问计数器（长度 25，记录每个状态的访问次数）
        cnt = [0 for i in range(25)]

        # 固定初始状态为 10（地图中部）
        initState = 10
        # 随机选择初始动作
        initAction = random.randint(0, 4)

        if trajectorySteps == -1:
            stop_when_reach_target = True  # 到达目标时自动停止

        # 按 epsilon-greedy 策略生成轨迹并计算每步奖励
        # 返回值：列表，每个元素为 (状态, 动作, 奖励, 下一状态, 下一动作) 五元组
        Trajectory = gridworld.getTrajectoryScore(
            nowState=initState,
            action=initAction,
            policy=policy_epsilon,
            steps=trajectorySteps,
            stop_when_reach_target=True)

        # 在轨迹末尾追加 sarsaStep 个目标格自循环，确保后 n 步的 Q 值更新有来源
        for i in range(sarsaStep):
            # (17, 4, 1, 17, 4)：目标格自循环（状态17，动作4，奖励1，仍在状态17）
            Trajectory.append((17, 4, 1, 17, 4))

        print("trajectorySteps", len(Trajectory))  # 打印轨迹总长度（含追加步）

        steps = len(Trajectory)  # 轨迹总长度 = 有效步数 + sarsaStep

        # 计算整条轨迹的折扣累积奖励（用于打印展示，不参与 Q 值更新）
        g = 1          # 折扣累积因子初始值
        reward = 0     # 累积奖励初始值
        for k in range(0, steps, 1):  # 遍历整条轨迹
            tmpstate, tmpaction, tmpscore, nextState, nextAction = Trajectory[k]
            reward += tmpscore * g  # 累加折扣奖励 r_k * γ^k
            g *= gamma              # 更新折扣因子
        print("Trajectory_score", reward)  # 打印轨迹总折扣奖励

        # n-step SARSA 核心更新循环
        # 对轨迹中每个有效时刻 t（从 0 到 steps-n），计算 n 步回报并更新 Q 值
        for k in range(0, steps - sarsaStep + 1, 1):
            cnt[Trajectory[k][0]] += 1  # 记录 t=k 时刻状态的访问次数

            # 初始化当前时刻的状态和动作（-1 表示尚未赋值）
            nowState, nowAction = -1, -1
            g = 1      # n 步折扣因子，初始为 1（即 γ^0）
            reward = 0  # n 步累积奖励初始为 0

            for i in range(sarsaStep):  # 向前展望 n 步
                # 解包轨迹中第 k+i 步的转移信息
                tmpstate, tmpaction, tmpscore, nextState, nextAction = Trajectory[k + i]

                # 仅在第一步（i=0）时记录起始状态和动作
                if nowState == -1:
                    nowState, nowAction = tmpstate, tmpaction  # 记录初始状态和动作

                # 累加折扣奖励：reward += r_{k+i} * γ^i
                reward += tmpscore * g
                # 更新折扣因子（下一步乘以 γ）
                g *= gamma

            # 加上第 n 步之后的自举值：G_t^(n) += γ^n * Q(s_{t+n}, a_{t+n})
            reward += g * action_value[nextState][nextAction]

            # n-step TD 误差：Q(s_t, a_t) - G_t^(n)
            TD_error = action_value[nowState][nowAction] - reward
            # 按 TD 误差更新 Q 值（朝减小误差方向修正）
            action_value[nowState][nowAction] -= learning_rate * TD_error

        # 策略改进：选 Q 值最大的动作作为新贪心策略
        # np.argmax(action_value, axis=1)：shape (25,)，每个状态的最优动作索引
        policy = np.eye(5)[np.argmax(action_value, axis=1)]  # shape (25, 5)
        # 更新 epsilon-greedy 策略
        policy_epsilon = np.vectorize(d.get)(policy)

        print(np.array(cnt).reshape(5, 5))  # 打印 5×5 状态访问次数矩阵

        # 计算状态价值 V(s) = Σ_a π(a|s)*Q(s,a)，shape (25,)
        state_value = np.sum(policy_epsilon * action_value, axis=1)
        # 全局平均状态价值
        mean_state_value = np.sum(policy_epsilon * action_value, axis=1).mean()

        gridworld.showPolicy(policy)                                   # 显示当前策略
        print(np.round(action_value, decimals=3).reshape(-1, 5))      # 打印 Q 表（精确到3位小数）
        print(np.round(state_value, decimals=4).reshape(5, 5))        # 打印 5×5 状态价值
        print("mean_state_value", mean_state_value)                    # 打印平均价值

    return gridworld  # 返回训练后的环境对象

## 三、实验：不同 n 步和折扣因子对比

### 3.1 n=3 步、gamma=0.99

In [5]:
# 使用 n=3 步、gamma=0.99 训练智能体
# n=3：每次使用 3 步累积奖励+第4步 Q 值自举，比 1 步 SARSA 偏差更小
n_step_SARSA(
    gridworld,
    sarsaStep=3,         # n=3 步
    trajectorySteps=5000, # 每条轨迹最多 5000 步
    final_epsilon=0.01,   # 最终探索率 1%
    num_episodes=600,     # 训练 600 轮
    gamma=0.99)           # 高折扣因子，重视长期奖励

episode 0/600
p1 0.6008 p0 0.0998
trajectorySteps 13
Trajectory_score -14.846976058494736
[[0 0 0 0 0]
 [0 0 0 0 0]
 [3 3 1 0 0]
 [2 0 2 0 0]
 [0 0 0 0 0]]
⬆️⬆️⬆️⬆️⬆️
⬆️⏫️⏫️⬆️⬆️
⬆️⬆️⏫️⬆️⬆️
⬆️⏫️✅⏫️⬆️
⬆️⏫️⬆️⬆️⬆️
[[ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.    -0.019  0.     0.    -0.019]
 [ 0.     0.    -0.008  0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.    -0.008  0.003]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.    

### 3.2 n=2 步（默认 gamma=0.9）

In [6]:
# 使用 n=2 步、默认 gamma=0.9 训练
# n=2 步比 n=3 收敛略快（步数少，每步更新更频繁）
n_step_SARSA(gridworld, sarsaStep=2, trajectorySteps=5000)  # 默认 gamma=0.9

episode 0/600
p1 0.6008 p0 0.0998
trajectorySteps 18
Trajectory_score -6.8329084688975446
[[0 0 0 0 0]
 [0 0 0 0 0]
 [6 3 0 0 0]
 [4 2 2 0 0]
 [0 0 0 0 0]]
⬆️⬆️⬆️⬆️⬆️
⬆️⏫️⏫️⬆️⬆️
⬆️⬆️⏫️⬆️⬆️
⬆️⏩️✅⏫️⬆️
⬆️⏫️⬆️⬆️⬆️
[[ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.    -0.001 -0.003 -0.001]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.    -0.019  0.    -0.012  0.   ]
 [ 0.     0.001  0.     0.    -0.009]
 [ 0.     0.     0.001  0.     0.002]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.    

### 3.3 折扣因子衰减对比（gamma=0.9 vs gamma=0.99）

In [ ]:
# 演示 gamma=0.9 时折扣因子随步数的衰减速度
# gamma^k：经 k 步折扣后的权重，体现对 k 步后奖励的重视程度
g = 1       # 初始折扣权重
res = []    # 存储每步的折扣权重值
for i in range(15):  # 展示前 15 步的折扣衰减
    g *= 0.9          # 每步乘以 gamma=0.9
    res.append(g * 100 // 1 / 100)  # 保留两位小数
print(res)  # 打印折扣权重列表（可见 gamma=0.9 衰减较快）

[0.9, 0.81, 0.72, 0.65, 0.59, 0.53, 0.47, 0.43, 0.38, 0.34, 0.31, 0.28, 0.25, 0.22, 0.2]


In [ ]:
# 演示 gamma=0.99 时折扣因子的衰减速度（比较 0.9 更慢）
g = 1       # 初始折扣权重
res = []    # 存储每步的折扣权重值
for i in range(15):  # 展示前 15 步的折扣衰减
    g *= 0.99          # 每步乘以 gamma=0.99
    res.append(g * 100 // 1 / 100)  # 保留两位小数
print(res)  # gamma=0.99 衰减更慢，说明对远期奖励更加重视

[0.99, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92, 0.91, 0.9, 0.89, 0.88, 0.87, 0.86, 0.86]
